In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import torch
import pandas as pd
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageDraw, ImageFont
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection


In [2]:
image_dir = "/fs/ess/PAS2136/CarabidImaging/Images/FinalImages/ABTrays/"

img_path = []

files = os.listdir(image_dir)


for img in files : 
    impath = os.path.join(image_dir, f"{img}")
    if os.path.exists(impath) : 
        img_path.append(impath)
        

In [3]:
csv_filepath = "/fs/ess/PAS2136/CarabidImaging/allIndividuals.csv"

df = pd.read_csv(csv_filepath)
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

df.head(3)


,uid,namedLocation,domainID,siteID,plotID,setDate,collectDate,identifiedDate,individualID,sampleCondition,...,ID_status,numbericID,yearCollected,scientificName_Species,imageID,Order,NumberOfBeetlesInTray,notes,processingNotes,imagePath
0,46ea6e01-8f09-4240-af5d-141629a9eab4,UKFS_003.basePlot.bet,D06,UKFS,UKFS_003,2019-07-09,2019-07-23,2020-10-19,NEON.BET.D06.004309,NaN,...,Expert,4309,2019,Calosoma scrutator,Calosoma_scrutator-Btray-Y2019-NEON.BET.D06.00...,1,10,NaN,NaN,/Images/FinalImages/ABTrays
1,3304e7fe-4dfe-4eb3-9a64-31505bbd9b34,UKFS_006.basePlot.bet,D06,UKFS,UKFS_006,2019-07-10,2019-07-24,2020-10-19,NEON.BET.D06.004314,NaN,...,Expert,4314,2019,Calosoma scrutator,Calosoma_scrutator-Btray-Y2019-NEON.BET.D06.00...,2,10,NaN,NaN,/Images/FinalImages/ABTrays
2,fa511b1a-0743-4964-ae4c-ccf5f00a89ae,UKFS_006.basePlot.bet,D06,UKFS,UKFS_006,2019-07-10,2019-07-24,2020-10-19,NEON.BET.D06.004315,NaN,...,Expert,4315,2019,Calosoma scrutator,Calosoma_scrutator-Btray-Y2019-NEON.BET.D06.00...,3,10,NaN,NaN,/Images/FinalImages/ABTrays


In [4]:

# def compute_iou(box1, box2):
    
#     x1, y1, x2, y2 = box1
#     x1g, y1g, x2g, y2g = box2
    
#     xi1 = max(x1, x1g)
#     yi1 = max(y1, y1g)
#     xi2 = min(x2, x2g)
#     yi2 = min(y2, y2g)
    
#     inter_width = max(0, xi2 - xi1)
#     inter_height = max(0, yi2 - yi1)
#     intersection = inter_width * inter_height
    
#     box1_area = (x2 - x1) * (y2 - y1)
#     box2_area = (x2g - x1g) * (y2g - y1g)
#     union = box1_area + box2_area - intersection
    
#     return intersection / union if union > 0 else 0


# def filter_overlapping_boxes(detections, iou_threshold=0.5):
    
#     if len(detections) <= 1:
#         return detections
        
#     sorted_detections = sorted(detections, key=lambda x: x['score'], reverse=True)
    
#     keep = []
    
#     for detection in sorted_detections:
#         box = detection['box']
#         should_keep = True
        
#         for kept_detection in keep:
#             kept_box = kept_detection['box']
#             if compute_iou(box, kept_box) > iou_threshold:
#                 should_keep = False
#                 break
        
#         if should_keep:
#             keep.append(detection)
    
#     return keep
    

# def detect_objects(image_paths, df, text="a beetle.", box_threshold=0.3, text_threshold=0.2, max_size_ratio=0.2, output_dir="detections"):
    
#     os.makedirs(output_dir, exist_ok=True)
    
#     model_id = "IDEA-Research/grounding-dino-base"
#     device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#     processor = AutoProcessor.from_pretrained(model_id)
#     model = AutoModelForZeroShotObjectDetection.from_pretrained(model_id).to(device)
    
#     results = {}
#     unmatched = []
#     count_records = []
    
#     for image_path in tqdm(image_paths[:5], desc="Processing ... "):
    
#         image = Image.open(image_path).convert("RGB")
#         img_width, img_height = image.size
#         img_area = img_width * img_height
#         base_name = os.path.basename(image_path)
        
        
#         image_rows = df[df['imageID'] == base_name]
        
#         if not image_rows.empty:
#             actual_count = int(image_rows['NumberOfBeetlesInTray'].iloc[0])
#         else:
#             actual_count = None
        
        
#         inputs = processor(images=image, text=text, return_tensors="pt").to(device)
        
#         with torch.no_grad():
#             outputs = model(**inputs)

        
#         detection_results = processor.post_process_grounded_object_detection(
#             outputs,
#             inputs.input_ids,
#             box_threshold=box_threshold,
#             text_threshold=text_threshold,
#             target_sizes=[image.size[::-1]]
#         )

        
#         filtered_boxes = []
        
#         for result in detection_results:
        
#             boxes = result["boxes"]
#             scores = result["scores"]
#             for box, score in zip(boxes, scores):
#                 x1, y1, x2, y2 = box.tolist()
#                 box_area = (x2 - x1) * (y2 - y1)
#                 size_ratio = box_area / img_area
#                 if size_ratio <= max_size_ratio:
#                     filtered_boxes.append({
#                         'box': [x1, y1, x2, y2],
#                         'score': score.item(),
#                         'size_ratio': size_ratio
#                     })
                    
#         filtered_boxes = filter_overlapping_boxes(filtered_boxes, iou_threshold=0.4)
        
#         results[image_path] = filtered_boxes

#         detected_count = len(filtered_boxes)

#         count_records.append({
#             "filename": base_name,
#             "actual": actual_count,
#             "detected": detected_count
#         })
        
#         if actual_count is not None and detected_count != actual_count:
#             unmatched.append({
#                 "filename": base_name,
#                 "actual": int(actual_count),
#                 "detected": int(detected_count)
#             })

        
#         if filtered_boxes:
        
#             draw_image = image.copy()
#             draw = ImageDraw.Draw(draw_image)
            
#             font_size = max(100, img_width // 20)
#             try:
#                 title_font = ImageFont.truetype("arial.ttf", font_size)
#             except:
#                 title_font = ImageFont.load_default()
            
#             for i, detection in enumerate(filtered_boxes):
                
#                 x1, y1, x2, y2 = detection['box']
#                 score = detection['score']
                
#                 draw.rectangle([x1, y1, x2, y2], outline="red", width=4)

#             actual_display = actual_count if actual_count is not None else "N/A"
#             print(f"{base_name:<110} => True: {actual_display:<3}, DINO: {detected_count:<3}")

#             base_name = os.path.splitext(os.path.basename(image_path))[0]
#             output_path = os.path.join(output_dir, f"{base_name}.png")
#             draw_image.save(output_path, format='PNG')
            
#         else:
#             print(f"No valid detections found: {base_name}")

#     if unmatched:
#         print("\nMismatched detections:")
#         for item in unmatched:
#             print(f"{item['filename']} => True: {item['actual']}, DINO: {item['detected']}")

    
#     count_df = pd.DataFrame(count_records)
    
#     return results, count_df
    

In [5]:
def compute_iou(box1, box2):
    x1, y1, x2, y2 = box1
    x1g, y1g, x2g, y2g = box2
    
    xi1 = max(x1, x1g)
    yi1 = max(y1, y1g)
    xi2 = min(x2, x2g)
    yi2 = min(y2, y2g)
    
    inter_width = max(0, xi2 - xi1)
    inter_height = max(0, yi2 - yi1)
    intersection = inter_width * inter_height
    
    box1_area = (x2 - x1) * (y2 - y1)
    box2_area = (x2g - x1g) * (y2g - y1g)
    union = box1_area + box2_area - intersection
    
    return intersection / union if union > 0 else 0


def compute_containment_ratio(small_box, large_box):

    sx1, sy1, sx2, sy2 = small_box
    lx1, ly1, lx2, ly2 = large_box
    
    # Find intersection
    xi1 = max(sx1, lx1)
    yi1 = max(sy1, ly1)
    xi2 = min(sx2, lx2)
    yi2 = min(sy2, ly2)
    
    # Calculate intersection area
    inter_width = max(0, xi2 - xi1)
    inter_height = max(0, yi2 - yi1)
    intersection = inter_width * inter_height
    
    # Calculate small box area
    small_area = (sx2 - sx1) * (sy2 - sy1)
    
    # Return containment ratio
    return intersection / small_area if small_area > 0 else 0



def filter_contained_boxes(detections, containment_threshold=0.5, size_ratio_threshold=0.75):

    if len(detections) <= 1:
        return detections
    
    sorted_detections = sorted(detections, key=lambda x: x['score'], reverse=True)
    keep = []
    
    for i, detection in enumerate(sorted_detections):
        current_box = detection['box']
        current_area = (current_box[2] - current_box[0]) * (current_box[3] - current_box[1])
        should_keep = True
        
        # Check against all boxes we're keeping
        for kept_detection in keep:
            kept_box = kept_detection['box']
            kept_area = (kept_box[2] - kept_box[0]) * (kept_box[3] - kept_box[1])
            
            # Check if current box is smaller and potentially contained
            if current_area < kept_area * size_ratio_threshold:
                containment_ratio = compute_containment_ratio(current_box, kept_box)
                if containment_ratio > containment_threshold:
                    should_keep = False
                    break
        
        if should_keep:
            keep.append(detection)
    
    return keep


def filter_overlapping_boxes(detections, iou_threshold=0.5):
    if len(detections) <= 1:
        return detections
        
    sorted_detections = sorted(detections, key=lambda x: x['score'], reverse=True)
    
    keep = []
    
    for detection in sorted_detections:
        box = detection['box']
        should_keep = True
        
        for kept_detection in keep:
            kept_box = kept_detection['box']
            if compute_iou(box, kept_box) > iou_threshold:
                should_keep = False
                break
        
        if should_keep:
            keep.append(detection)
    
    return keep


def detect_objects(image_paths, df, text="a beetle.", box_threshold=0.3, text_threshold=0.2, max_size_ratio=0.2, output_dir="detections"):
    
    os.makedirs(output_dir, exist_ok=True)
    
    model_id = "IDEA-Research/grounding-dino-base"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    processor = AutoProcessor.from_pretrained(model_id)
    model = AutoModelForZeroShotObjectDetection.from_pretrained(model_id).to(device)
    
    results = {}
    unmatched = []
    count_records = []
    
    for image_path in tqdm(image_paths, desc="Processing ... "):
    
        image = Image.open(image_path).convert("RGB")
        img_width, img_height = image.size
        img_area = img_width * img_height
        base_name = os.path.basename(image_path)
        
        image_rows = df[df['imageID'] == base_name]
        
        if not image_rows.empty:
            actual_count = int(image_rows['NumberOfBeetlesInTray'].iloc[0])
        else:
            actual_count = None
        
        inputs = processor(images=image, text=text, return_tensors="pt").to(device)
        
        with torch.no_grad():
            outputs = model(**inputs)

        detection_results = processor.post_process_grounded_object_detection(
            outputs,
            inputs.input_ids,
            box_threshold=box_threshold,
            text_threshold=text_threshold,
            target_sizes=[image.size[::-1]]
        )

        # Initial filtering by size
        filtered_boxes = []
        
        for result in detection_results:
            boxes = result["boxes"]
            scores = result["scores"]
            for box, score in zip(boxes, scores):
                x1, y1, x2, y2 = box.tolist()
                box_area = (x2 - x1) * (y2 - y1)
                size_ratio = box_area / img_area
                if size_ratio <= max_size_ratio:
                    filtered_boxes.append({
                        'box': [x1, y1, x2, y2],
                        'score': score.item(),
                        'size_ratio': size_ratio
                    })
        
        # Apply traditional NMS first
        filtered_boxes = filter_overlapping_boxes(filtered_boxes, iou_threshold=0.4)
        
        # Apply containment filtering to remove small boxes inside larger ones
        filtered_boxes = filter_contained_boxes(
            filtered_boxes, 
            containment_threshold=0.6,  # {x}% of small box must be inside large box
            size_ratio_threshold=0.75   # Small box must be <{x}% size of large box
        )
        
        results[image_path] = filtered_boxes

        detected_count = len(filtered_boxes)

        count_records.append({
            "filename": base_name,
            "actual": actual_count,
            "detected": detected_count
        })
        
        if actual_count is not None and detected_count != actual_count:
            unmatched.append({
                "filename": base_name,
                "actual": int(actual_count),
                "detected": int(detected_count)
            })

        if filtered_boxes:
            
            draw_image = image.copy()
            draw = ImageDraw.Draw(draw_image)
            
            for i, detection in enumerate(filtered_boxes):
                x1, y1, x2, y2 = detection['box']
                score = detection['score']
                
                draw.rectangle([x1, y1, x2, y2], outline="red", width=4)

            actual_display = actual_count if actual_count is not None else "N/A"
            print(f"{base_name:<110} => True: {actual_display:<3}, DINO: {detected_count:<3}")

            base_name = os.path.splitext(os.path.basename(image_path))[0]
            output_path = os.path.join(output_dir, f"{base_name}.png")
            draw_image.save(output_path, format='PNG')
            
        else:
            print(f"No valid detections found: {base_name}")

    if unmatched:
        print("\nMismatched detections:")
        for item in unmatched:
            print(f"{item['filename']} => True: {item['actual']}, DINO: {item['detected']}")

    count_df = pd.DataFrame(count_records)
    
    return results, count_df

In [ ]:

outdir = "GDINO-Detections-v4"

detections, count_df = detect_objects(img_path, df, text="a beetle.", max_size_ratio=0.05, output_dir=outdir)    


Processing ... :   0%|          | 0/731 [00:00<?, ?it/s]

Carabus_serratus-Btray-Y2018-NEON.BET.D09.003068-NEON.BET.D09.003900.png                                       => True: 13 , DINO: 13 
Pterostichus_permundus-Btray-Y2021-NEON.BET.D02.010685-NEON.BET.D02.010876.png                                 => True: 23 , DINO: 23 
Sphaeroderus_canadensis-Btray-Y2021-NEON.BET.D01.006860-NEON.BET.D01.006900.png                                => True: 25 , DINO: 25 
Calosoma_peregrinator-Btray-Y2021-NEON.BET.D14.001771-NEON.BET.D14.001811.png                                  => True: 12 , DINO: 12 
Cyclotrachelus_fucatus-Btray-Y2018-NEON.BET.D07.003398-NEON.BET.D07.005282.png                                 => True: 31 , DINO: 31 
Chlaenius_aestivus-Atray-Y2022-NEON.BET.D02.002930-NEON.BET.D02.13253.png                                      => True: 72 , DINO: 72 
Scaphinotus_angusticollis-Atray-Y2018-NEON.BET.D16.001169-NEON.BET.D16.001882.png                              => True: 42 , DINO: 42 
Carabus_truncaticollis-Atray-Y2021-NEON.BET.D18.002723-

In [ ]:
count_df.head(5)

In [ ]:

outfileName = f"{outdir}/GDino-Detection-Stats.csv"

count_df.to_csv(outfileName, index=False)
